# Пример 5 из 5: регрессия версий и эксплуатационные метрики — тема 16

Зрелый eval помогает принимать решения об изменениях: **регрессионное** сравнение версий агента и баланс **качество — стоимость — время**. Сравниваем две версии агента, считаем эксплуатационные метрики, задаём **пороги качества (go/no-go)** и разбираем **хвосты** (провальные кейсы) как источник новых задач.

In [1]:
import os, json
import pandas as pd
from eval_harness import run_suite, grade_state, grade_safety
import eval_env as E

STRONG = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")
LIGHT  = os.environ.get("LLM_MODEL_CHEAP", "openai/gpt-4.1-nano")
suite = json.load(open("data/eval_suite.json"))["tasks"]
print("Корзина:", len(suite), "| STRONG:", STRONG, "| LIGHT:", LIGHT)

Корзина: 16 | STRONG: openai/gpt-4.1-mini | LIGHT: openai/gpt-4.1-nano


## Регрессия: версия V1 против V2

Две версии агента отличаются системным промптом: **V1** — упрощённый, **V2** — с явной проверкой политики перед изменением заказа. Прогоняем обе на одной корзине и в одном контуре оценки.

In [2]:
def metrics(runs, tasks):
    n = len(tasks)
    state = sum(grade_state(t, runs[t["id"]]) for t in tasks) / n
    errs  = sum(runs[t["id"]].error != "" for t in tasks) / n
    sec   = sum(runs[t["id"]].seconds for t in tasks) / n
    toks  = sum(runs[t["id"]].tokens for t in tasks) / n
    return {"state_pass": round(state, 2), "error_rate": round(errs, 2),
            "avg_sec": round(sec, 1), "avg_tokens": int(toks)}

runs_v1 = run_suite(suite, E.SYSTEM_V1, model=STRONG)
runs_v2 = run_suite(suite, E.SYSTEM_V2, model=STRONG)

reg = pd.DataFrame([{"версия": "V1 (упрощённый)", **metrics(runs_v1, suite)},
                    {"версия": "V2 (с проверкой политики)", **metrics(runs_v2, suite)}])
print(reg.to_string(index=False))

                   версия  state_pass  error_rate  avg_sec  avg_tokens
          V1 (упрощённый)        0.94         0.0      6.1         803
V2 (с проверкой политики)        1.00         0.0      7.1        1118


## Экономика: та же версия на лёгкой и мощной модели

Помимо качества важна **стоимость**. Прогоняем V2 на **лёгкой** модели против мощной на подвыборке и сравниваем качество/токены/время. Дифференцированный подбор модели — способ снизить стоимость там, где хватает лёгкой.

In [3]:
sub = suite[:8]
runs_strong = run_suite(sub, E.SYSTEM_V2, model=STRONG)
runs_light  = run_suite(sub, E.SYSTEM_V2, model=LIGHT)

econ = pd.DataFrame([{"модель": STRONG, **metrics(runs_strong, sub)},
                     {"модель": LIGHT,  **metrics(runs_light, sub)}])
print(econ.to_string(index=False))

             модель  state_pass  error_rate  avg_sec  avg_tokens
openai/gpt-4.1-mini         1.0         0.0      6.3        1007
openai/gpt-4.1-nano         1.0         0.0      5.9        1009


## Пороги качества (go / no-go)

Для релиза задают **жёсткие пороги** — удобные no-go условия. Проверяем каждую версию против порогов (нахождение самих порогов — тоже часть построения eval).

In [4]:
THRESHOLDS = {"state_pass": 0.80, "error_rate": 0.05, "avg_sec": 15.0}

def gate(m):
    ok = (m["state_pass"] >= THRESHOLDS["state_pass"] and
          m["error_rate"] <= THRESHOLDS["error_rate"] and
          m["avg_sec"]   <= THRESHOLDS["avg_sec"])
    return "GO" if ok else "NO-GO"

for name, runs in [("V1", runs_v1), ("V2", runs_v2)]:
    m = metrics(runs, suite)
    print(f"{name}: {m} -> {gate(m)}")

V1: {'state_pass': 0.94, 'error_rate': 0.0, 'avg_sec': 6.1, 'avg_tokens': 803} -> GO
V2: {'state_pass': 1.0, 'error_rate': 0.0, 'avg_sec': 7.1, 'avg_tokens': 1118} -> GO


## Хвосты распределения: провальные кейсы

Хвосты важнее среднего: каждый провал — это либо ошибка агента, либо кандидат в **новую задачу корзины** (замыкание цикла eval через логи и баги). Выведем задачи, где V2 не прошёл state-проверку.

In [5]:
fails = [{"id": t["id"], "cat": t["category"], "diff": t["difficulty"],
          "tools": runs_v2[t["id"]].tool_calls, "err": runs_v2[t["id"]].error[:40]}
         for t in suite if not grade_state(t, runs_v2[t["id"]])]
if fails:
    print("Провальные кейсы V2 (кандидаты в пополнение корзины):")
    print(pd.DataFrame(fails).to_string(index=False))
else:
    print("V2 прошёл state-проверку на всех задачах корзины.")

V2 прошёл state-проверку на всех задачах корзины.


## Итоги

- **Регрессионный прогон** двух версий в одном контуре показывает, стало **лучше/хуже/иначе** по качеству, стоимости и времени — решение принимается по данным.
- **Дифференцированный подбор модели** позволяет экономить: лёгкая модель может быть достаточной там, где не нужна максимальная мощность.
- **Пороги качества (go/no-go)** превращают eval в инструмент управления релизами.
- **Хвосты** (провальные кейсы) — главный источник роста корзины; так замыкается цикл eval.

**Общий вывод модуля:** eval — это инженерная система из корзины задач, инфраструктуры, критериев и грейдинга; оценивается вся траектория и итоговое состояние среды, стабильность (pass^k) и баланс «качество — стоимость — время», а сам процесс цикличен.